In [ ]:
#In bronze we are reading in the json raw data and returning 2 parquet files playlist and info

#write paruqet files to azure in parallel
#spark can read all parquet files at once as a dataframe

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode


# ============================================================
# 1. Create Spark session
# ============================================================

spark = (
    SparkSession.builder
    .appName("Spotify Bronze ETL")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-azure:3.4.2"
    )
    .config(
        "fs.azure.account.auth.type.spotifydestorage.dfs.core.windows.net",
        "OAuth"
    )
    .config(
        "fs.azure.account.oauth.provider.type.spotifydestorage.dfs.core.windows.net",
        "org.apache.hadoop.fs.azurebfs.oauth2.MsiTokenProvider"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


# ============================================================
# 2. Azure Storage paths
# ============================================================

STORAGE_ACCOUNT = "spotifydestorage"

RAW_CONTAINER = "raw-json-data"
BRONZE_CONTAINER = "data"

RAW_PATH = (
    f"abfss://{RAW_CONTAINER}@"
    f"{STORAGE_ACCOUNT}.dfs.core.windows.net/"
)

BRONZE_PATH = (
    f"abfss://{BRONZE_CONTAINER}@"
    f"{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    "bronze/"
)


# ============================================================
# 3. Read all JSON slices
# ============================================================

print("========================================")
print("Reading JSON slices from Azure...")
print("========================================")

raw_df = (
    spark.read
    .option("multiLine", True)
    .json(RAW_PATH)
)

print("JSON read successful.")
print("Top-level columns:")
print(raw_df.columns)


# ============================================================
# 4. Create INFO DataFrame
# ============================================================

print("\n========================================")
print("Creating info DataFrame...")
print("========================================")

info_df = raw_df.select("info.*")

print("Info schema:")
info_df.printSchema()


# ============================================================
# 5. Create PLAYLISTS DataFrame
# ============================================================

print("\n========================================")
print("Creating playlists DataFrame...")
print("========================================")

playlists_df = (
    raw_df
    .select(explode("playlists").alias("playlist"))
    .select("playlist.*")
)

print("Playlist schema:")
playlists_df.printSchema()


# ============================================================
# 6. Write INFO Parquet
# ============================================================

print("\n========================================")
print("Writing info Parquet...")
print("========================================")


info_output_path = BRONZE_PATH + "info.parquet"

(
    info_df
    .write
    .mode("overwrite")
    .parquet(info_output_path)
)

print(f"Info Parquet written to:")
print(info_output_path)


# ============================================================
# 7. Write PLAYLISTS Parquet
# ============================================================

print("\n========================================")
print("Writing playlists Parquet...")
print("========================================")

playlists_output_path = BRONZE_PATH + "playlists.parquet"

(
    playlists_df
    .write
    .mode("overwrite")
    .parquet(playlists_output_path)
)

print(f"Playlists Parquet written to:")
print(playlists_output_path)


# ============================================================
# 8. Final summary
# ============================================================

print("\n========================================")
print("BRONZE ETL COMPLETE")
print("========================================")

print("Info output:")
print(info_output_path)

print("\nPlaylists output:")
print(playlists_output_path)


# ============================================================
# 9. Stop Spark
# ============================================================

spark.stop()



:: loading settings :: url = jar:file:/home/linux_vm_user/spotifymp-project/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/linux_vm_user/.ivy2.5.2/cache
The jars for the packages stored in: /home/linux_vm_user/.ivy2.5.2/jars
org.apache.hadoop#hadoop-azure added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-835dc11a-109b-47f4-8b2d-193db333759c;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-azure;3.4.2 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.3.0 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-guava;1.4.0 in central
	found org.eclipse.jetty#

Reading JSON slices from Azure...


JSON read successful.
Top-level columns:
['info', 'playlists']

Creating info DataFrame...
Info schema:
root
 |-- generated_on: string (nullable = true)
 |-- slice: string (nullable = true)
 |-- version: string (nullable = true)


Creating playlists DataFrame...
Playlist schema:
root
 |-- collaborative: string (nullable = true)
 |-- description: string (nullable = true)
 |-- duration_ms: long (nullable = true)
 |-- modified_at: long (nullable = true)
 |-- name: string (nullable = true)
 |-- num_albums: long (nullable = true)
 |-- num_artists: long (nullable = true)
 |-- num_edits: long (nullable = true)
 |-- num_followers: long (nullable = true)
 |-- num_tracks: long (nullable = true)
 |-- pid: long (nullable = true)
 |-- tracks: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- album_name: string (nullable = true)
 |    |    |-- album_uri: string (nullable = true)
 |    |    |-- artist_name: string (nullable = true)
 |    |    |-- artist_uri: strin

Info Parquet written to:
abfss://data@spotifydestorage.dfs.core.windows.net/bronze/info.parquet

Writing playlists Parquet...


Playlists Parquet written to:
abfss://data@spotifydestorage.dfs.core.windows.net/bronze/playlists.parquet

BRONZE ETL COMPLETE
Info output:
abfss://data@spotifydestorage.dfs.core.windows.net/bronze/info.parquet

Playlists output:
abfss://data@spotifydestorage.dfs.core.windows.net/bronze/playlists.parquet
